# MQTT class recommendation: flat representations

This notebook compares whole-payload text representations using the same embedding model as the backend. Evaluation is **vendor-held-out class recommendation**, not KMeans clustering: all streams from the query stream's manufacturer are excluded when class prototypes are built.

The dataset is intentionally small (29 streams), so these results are exploratory rather than publication-grade.

In [ ]:
from pathlib import Path
import sys

workdir = Path.cwd()
notebook_dir = workdir if (workdir / "class_recommendation_eval.py").exists() else workdir / "notebook"
sys.path.insert(0, str(notebook_dir.resolve()))

from class_recommendation_eval import (
    DEFAULT_MODEL,
    dataset_summary,
    load_dataset,
    load_model,
    run_flat_experiment,
)


## Dataset validation

`load_dataset()` rejects duplicate topics, payload/metadata mismatches, missing labels, non-object payloads, and classes that cannot support vendor-held-out evaluation.

In [ ]:
data = load_dataset()
print(f"Validated {len(data)} streams, {data['true_class'].nunique()} classes, {data['manufacturer'].nunique()} manufacturers")
display(dataset_summary(data))


## Encode and evaluate

Representations:

- value only;
- key only;
- `key: value`;
- categorical `key: value` with numeric values removed;
- field schema (`numeric` or `categorical`).

Metrics are Top-1 accuracy, Macro-F1, Recall@3, and mean reciprocal rank (MRR).

In [ ]:
MODEL_NAME = DEFAULT_MODEL
print("Loading", MODEL_NAME)
model = load_model(MODEL_NAME)
flat_metrics, flat_cases = run_flat_experiment(data, model)
display(flat_metrics)


## Inspect the best method case by case

Do not select a production strategy from the aggregate score alone. Inspect which classes and payload patterns fail.

In [ ]:
best_flat = flat_metrics.iloc[0]['method']
best_flat_cases = flat_cases[flat_cases['method'] == best_flat].copy()
print("Best flat method:", best_flat)
display(best_flat_cases.sort_values(['correct', 'true_class', 'topic']))
display(
    best_flat_cases.groupby('true_class')['correct']
    .agg(['count', 'sum', 'mean'])
    .rename(columns={'sum': 'correct', 'mean': 'class_accuracy'})
)
